# Regressione logistica

In [13]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from pathlib import Path
import warnings
# Nascondo i warning
warnings.filterwarnings('ignore')

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare treining
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Training

In [14]:
def training(file_path, csv_name):
    # Leggo i csv
    df = pd.read_csv(file_path)

    # Filtro solo le pazienti con PR valido
    df_PRvalido = df[df['PR [SII]'].notna()].copy()

    # Vado a separare le features e target
    features = df_PRvalido.drop(columns=['Patient ID', 'lesion idx', 'tumor/benign', 
                             'GRADE', 'ER [SII]', 'PR [SII]', 'HER2 [SII]', 
                             'isTN', 'KI67 [%]', 'Breast'])

    # Prendo solo PR [SII], convertita in valori interi.
    # LogisticRegression in sklearn accetta solo target discrete.
    target = df_PRvalido['PR [SII]'].astype(int)

    # Normalizzo i dati per evitare problemi di scala e overflow
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    # Istanzio il modello di regressione logistica con solver robusto e molte iterazioni
    logreg = LogisticRegression(max_iter=50000, random_state=42, solver='saga', tol=1e-3)

    # Alleno il modello sui dati normalizzati
    logreg.fit(features_scaled, target)

    # Creo la cross-validation a 5 fold stratificata
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(logreg, features_scaled, target, cv=cv, scoring='accuracy')
    
    # Ritorno risultati sintetici: media, deviazione standard e score per ogni fol
    return {
        'mean_accuracy': scores.mean(),     #Accuratezza media su tutte le fold
        'std_accuracy': scores.std(),       #Variabilità tra le fold
        'scores_per_fold': scores           #Accuratezza tra ciascun fold
    }

# Lettura dei file

In [15]:
results = {}
print("="*50 +"\nTRAINING REGRESSIONE LOGISTICA\n" + "="*50)
results = {}
for name, file_path in datasets.items():
    print(f"\n{name}")
    results[name] = training(file_path, name)
    print(f"Accuracy media: {results[name]['mean_accuracy']:.3f} ± {results[name]['std_accuracy']:.3f}")
    print(f"Scores per fold: {[f'{s:.3f}' for s in results[name]['scores_per_fold']]}")

TRAINING REGRESSIONE LOGISTICA

t2_medsam
Accuracy media: 0.416 ± 0.092
Scores per fold: ['0.571', '0.357', '0.385', '0.462', '0.308']

t2_preprocessed
Accuracy media: 0.346 ± 0.107
Scores per fold: ['0.143', '0.357', '0.462', '0.385', '0.385']

t2_original
Accuracy media: 0.316 ± 0.097
Scores per fold: ['0.214', '0.214', '0.385', '0.462', '0.308']

medsam_dynamic
Accuracy media: 0.315 ± 0.117
Scores per fold: ['0.214', '0.286', '0.231', '0.538', '0.308']

preprocessed_dynamic
Accuracy media: 0.314 ± 0.115
Scores per fold: ['0.286', '0.286', '0.231', '0.538', '0.231']

original_dynamic
Accuracy media: 0.330 ± 0.152
Scores per fold: ['0.214', '0.357', '0.231', '0.615', '0.231']
